# HexaDepth · Train a YOLO detector on MILCO side-scan sonar (Batch D2)

**Before you start:** *Runtime → Change runtime type → GPU (T4 is fine)*. Then run the cells **top to bottom**.

1. Cell 1 checks the GPU, cell 2 installs Ultralytics.
2. Cell 3 asks you to upload `yolo_milco.zip` (built locally with `python scripts/make_yolo_dataset.py`, ~110 MB).
3. Cell 5 trains YOLOv8n (pretrained) at `imgsz=1024`, 80 epochs, early stop after 20 epochs without improvement, fixed seed.
4. Cell 6 prints the validation metrics, cell 7 downloads `best.pt` (plus the training log) to your computer.

**Split:** train/val pool = 2010 + 2015 + 2017; val is a stratified ~15% random slice (by survey year × image type: contains NOMBO / MILCO only / empty; seed 0), train is the rest. The TEST years (2018, 2021) are **not** in the zip, so the model never sees them.

**Data credit:** Santos, Moura, Torgal, Lobo & Neto (2024), *Side-scan sonar imaging data of underwater vehicles for mine detection*, https://doi.org/10.6084/m9.figshare.24574879, CC BY 4.0.

### 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > GPU"

### 2 · Install Ultralytics

In [ ]:
!pip install -q "ultralytics>=8.3"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU runtime: Runtime > Change runtime type > GPU"

### 3 · Upload `yolo_milco.zip`
A file picker appears below. Choose `build/yolo_milco.zip` from the HexaDepth folder.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert "yolo_milco.zip" in uploaded, f"Expected yolo_milco.zip, got {list(uploaded)}"

### 4 · Unzip and point `data.yaml` at the dataset

In [ ]:
import pathlib, re, json
!rm -rf /content/yolo_milco && unzip -q -o /content/yolo_milco.zip -d /content
root = pathlib.Path("/content/yolo_milco")
yaml_path = root / "data.yaml"
yaml_path.write_text(re.sub(r"(?m)^path:.*$", f"path: {root}", yaml_path.read_text()))
print(yaml_path.read_text())
m = json.loads((root / "manifest.json").read_text())
for split, s in m["splits"].items():
    print(f"{split}: {s['images']} images by year {s['images_by_year']}, ({s['negative_images']} negatives), objects {s['objects']}")
print("TEST years excluded:", m["test_years_excluded"])
for split in ("train", "val"):
    n_img = len(list((root / "images" / split).glob("*.jpg")))
    n_lbl = len(list((root / "labels" / split).glob("*.txt")))
    print(f"{split}: {n_img} images, {n_lbl} label files")

### 5 · Train YOLOv8n (pretrained), imgsz 1024, 80 epochs, patience 20, seed 0
Takes roughly 15–40 minutes on a T4. Output goes to `runs/detect/train/`.

In [ ]:
from ultralytics import YOLO
SEED = 0
model = YOLO("yolov8n.pt")                      # COCO-pretrained weights
results = model.train(
    data=str(yaml_path),
    imgsz=1024,
    epochs=80,
    patience=20,
    seed=SEED,
    deterministic=True,
    batch=16,
    project="/content/runs/detect",
    name="train",
    exist_ok=True,
)

### 6 · Validation metrics (stratified 15% val slice)
Copy this output back to the HexaDepth chat.

In [ ]:
best = "/content/runs/detect/train/weights/best.pt"
val = YOLO(best).val(data=str(yaml_path), imgsz=1024, split="val", seed=SEED, plots=False)
print(f"mAP50     {val.box.map50:.4f}")
print(f"mAP50-95  {val.box.map:.4f}")
print(f"precision {val.box.mp:.4f}")
print(f"recall    {val.box.mr:.4f}")
for i, name in val.names.items():
    p, r, ap50, ap = val.box.class_result(i)
    print(f"  {name:6s} P {p:.4f}  R {r:.4f}  mAP50 {ap50:.4f}  mAP50-95 {ap:.4f}")

### 7 · Download `best.pt` (and the training log)
Save `best.pt` into the HexaDepth repo as `models/yolo_milco_best.pt`.

In [ ]:
from google.colab import files
files.download("/content/runs/detect/train/weights/best.pt")
files.download("/content/runs/detect/train/results.csv")
files.download("/content/runs/detect/train/args.yaml")